<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам конструктора классов с использованием гетторов и сетторов и реализуйте взаимодействие объектов между собой

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

public class Item
{
    public string Name { get; private set; }
    public int Volume { get; private set; }
    public string Category { get; private set; }

    public Item(string name, int volume, string category = "Общая")
    {
        Name = name;
        Volume = volume;
        Category = category;
    }

    public override string ToString()
    {
        return $"[Товар: {Name}, Объем: {Volume}, Категория: {Category}]";
    }
}

public abstract class Inventory
{
    public int WarehouseId { get; private set; }
    public string WarehouseName { get; private set; }
    public int StorageCapacity { get; private set; }
    
    protected List<Item> Items { get; private set; }

    public int CurrentVolume => Items.Sum(item => item.Volume);

    protected Inventory(int warehouseId, string warehouseName, int storageCapacity)
    {
        WarehouseId = warehouseId;
        WarehouseName = warehouseName;
        StorageCapacity = storageCapacity;
        Items = new List<Item>();
    }

    public virtual bool CanAddItem(Item item)
    {
        return CurrentVolume + item.Volume <= StorageCapacity;
    }

    public virtual bool AddItem(Item item)
    {
        if (CanAddItem(item))
        {
            Items.Add(item);
            Console.WriteLine($"[Склад '{WarehouseName}'] Товар '{item.Name}' успешно добавлен.");
            return true;
        }
        Console.WriteLine($"[Склад '{WarehouseName}'] Ошибка: Недостаточно места для товара '{item.Name}'.");
        return false;
    }

    public virtual bool RemoveItem(Item item)
    {
        if (Items.Contains(item))
        {
            Items.Remove(item);
            Console.WriteLine($"[Склад '{WarehouseName}'] Товар '{item.Name}' успешно удален.");
            return true;
        }
        Console.WriteLine($"[Склад '{WarehouseName}'] Ошибка: Товар '{item.Name}' не найден на складе.");
        return false;
    }

    // --- ВЗАИМОДЕЙСТВИЕ ОБЪЕКТОВ ---
    public virtual bool TransferItem(Item item, Inventory targetWarehouse)
    {
        if (this.Items.Contains(item) && targetWarehouse.CanAddItem(item))
        {
            this.RemoveItem(item);
            targetWarehouse.AddItem(item);
            Console.WriteLine($"[Трансфер] Товар '{item.Name}' успешно перемещен со склада '{this.WarehouseName}' на '{targetWarehouse.WarehouseName}'.");
            return true;
        }
        Console.WriteLine($"[Трансфер] Не удалось переместить '{item.Name}'. Проверьте наличие товара и вместимость склада '{targetWarehouse.WarehouseName}'.");
        return false;
    }

    public List<Item> GetItemsByCategory(string category)
    {
        return Items.Where(i => i.Category.Equals(category, StringComparison.OrdinalIgnoreCase)).ToList();
    }

    public virtual void GetStorageStatus()
    {
        int freeSpace = StorageCapacity - CurrentVolume;
        Console.WriteLine($"--- Статус склада '{WarehouseName}' (ID: {WarehouseId}) ---");
        Console.WriteLine($"Вместимость: {StorageCapacity} | Занято: {CurrentVolume} | Свободно: {freeSpace}");
    }
}

public class PersonalInventory : Inventory
{
    public string OwnerName { get; private set; }

    public PersonalInventory(int warehouseId, string warehouseName, int storageCapacity, string ownerName)
        : base(warehouseId, warehouseName, storageCapacity)
    {
        OwnerName = ownerName;
    }

    public override void GetStorageStatus()
    {
        base.GetStorageStatus(); 
        Console.WriteLine($"Владелец склада: {OwnerName}");
        Console.WriteLine("---------------------------------------------");

    }
}

public class GroupInventory : Inventory
{
    public string ProductGroup { get; private set; }

    public GroupInventory(int warehouseId, string warehouseName, int storageCapacity, string productGroup)
        : base(warehouseId, warehouseName, storageCapacity)
    {
        ProductGroup = productGroup;
    }

    public override bool AddItem(Item item)
    {
        if (!item.Category.Equals(ProductGroup, StringComparison.OrdinalIgnoreCase))
        {
            Console.WriteLine($"[Групповой склад] Ошибка: Товар '{item.Name}' (Категория: {item.Category}) не соответствует группе склада '{ProductGroup}'.");
            return false;
        }

        if (base.AddItem(item))
        {
            Console.WriteLine($"[Групповой склад] Товар '{item.Name}' успешно отнесен к товарной группе '{ProductGroup}'.");
            return true;
        }
        return false;
    }
}

public class AutomatedInventory : Inventory
{
    public string AutomationLevel { get; private set; }

    public AutomatedInventory(int warehouseId, string warehouseName, int storageCapacity, string automationLevel)
        : base(warehouseId, warehouseName, storageCapacity)
    {
        AutomationLevel = automationLevel;
    }

    public override bool RemoveItem(Item item)
    {
        int volumeBefore = CurrentVolume;

        if (base.RemoveItem(item))
        {
            Console.WriteLine($"[Автоматизированный склад] Уровень автоматизации: {AutomationLevel}. Робот-погрузчик завершил операцию удаления товара '{item.Name}'.");
            return true;
        }
        return false;
    }
}

In [7]:
Item item1 = new Item("Ноутбук", 10, "Электроника");
Item item2 = new Item("Яблоко", 2, "Продукты");
Item item3 = new Item("Монитор", 15, "Электроника");
Item item4 = new Item("Банан", 1, "Продукты");

Inventory personalWarehouse = new PersonalInventory(1, "Личный склад", 50, "Иванов И.И.");
Inventory groupWarehouse = new GroupInventory(2, "Склад Продуктов", 100, "Продукты");
Inventory autoWarehouse = new AutomatedInventory(3, "Роботизированный комплекс", 200, "Робот-сортировщик v2.0");

Console.WriteLine("=== 1. БАЗОВОЕ ДОБАВЛЕНИЕ ===\n");
personalWarehouse.AddItem(item1);
groupWarehouse.AddItem(item2);
groupWarehouse.AddItem(item4);
autoWarehouse.AddItem(item3);

Console.WriteLine("\n=== 2. ДЕМОНСТРАЦИЯ ВЗАИМОДЕЙСТВИЯ (ТРАНСФЕР) ===\n");
groupWarehouse.TransferItem(item2, autoWarehouse); 

personalWarehouse.TransferItem(item1, autoWarehouse);

Console.WriteLine("\n=== 3. ДЕМОНСТРАЦИЯ ВЗАИМОДЕЙСТВИЯ (ФИЛЬТРАЦИЯ) ===\n");
List<Item> electronics = autoWarehouse.GetItemsByCategory("Электроника");
Console.WriteLine($"Электроника на автоматизированном складе: {string.Join(", ", electronics)}");

Console.WriteLine("\n=== 4. ИТОГОВЫЕ СТАТУСЫ ===\n");
personalWarehouse.GetStorageStatus();
groupWarehouse.GetStorageStatus();
autoWarehouse.GetStorageStatus();


=== 1. БАЗОВОЕ ДОБАВЛЕНИЕ ===

[Склад 'Личный склад'] Товар 'Ноутбук' успешно добавлен.
[Склад 'Склад Продуктов'] Товар 'Яблоко' успешно добавлен.
[Групповой склад] Товар 'Яблоко успешно отнесен к товарной группе 'Продукты'.
[Склад 'Склад Продуктов'] Товар 'Банан' успешно добавлен.
[Групповой склад] Товар 'Банан успешно отнесен к товарной группе 'Продукты'.
[Склад 'Роботизированный комплекс'] Товар 'Монитор' успешно добавлен.

=== 2. ДЕМОНСТРАЦИЯ ВЗАИМОДЕЙСТВИЯ (ТРАНСФЕР) ===

[Склад 'Склад Продуктов'] Товар 'Яблоко' успешно удален.
[Склад 'Роботизированный комплекс'] Товар 'Яблоко' успешно добавлен.
[Трансфер] Товар 'Яблоко' успешно перемещен со склада 'Склад Продуктов' на 'Роботизированный комплекс'.
[Склад 'Личный склад'] Товар 'Ноутбук' успешно удален.
[Склад 'Роботизированный комплекс'] Товар 'Ноутбук' успешно добавлен.
[Трансфер] Товар 'Ноутбук' успешно перемещен со склада 'Личный склад' на 'Роботизированный комплекс'.

=== 3. ДЕМОНСТРАЦИЯ ВЗАИМОДЕЙСТВИЯ (ФИЛЬТРАЦИЯ) ===

Электро